# Markov-switching AR(1) (Hamilton-style) vs TimesFM 2.5

**Goal:** Compare one-step-ahead forecasts to **TimesFM** when the data come from a **two-state Markov chain** that switches the **AR(1)** coefficient (a small **Hamilton / regime-switching** DGP).

**Setting (same window as AR / MA / ARMA notebooks):** **$n=100$**, first forecast at **$k=11$** (**89** test points).

**DGP (readable):**  
- Draw a Markov chain $s_t \in \{0,1\}$ with a fixed **2×2 transition matrix**.  
- Then $y_t = \phi_{s_t} y_{t-1} + \sigma_{s_t} \eta_t$ with i.i.d. standard normal $\eta_t$.

**Classical baseline:** **OLS AR(1)** fit on each expanding history (predict $a + b \, y_{t}$ from regressing $y_{1:}$ on $y_{:-1}$).  
**Why not statsmodels `MarkovAutoregression` for the forecast:** in current **statsmodels**, out-of-sample **`forecast` / `predict`** for these models raise **`NotImplementedError`**, so we use a transparent **AR(1)** benchmark instead (misspecified vs the true switching DGP, but fast and reproducible).

## 1. Setup

**Hugging Face:** `HF_TOKEN` — `pip install -e ".[experiments]"`, `.env`, or `export HF_TOKEN=...`.

Imports, repo path, `load_dotenv`, **random seed**.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="statsmodels")

HERE = Path.cwd().resolve()
REPO = HERE if (HERE / "src" / "timesfm").is_dir() else HERE.parent
try:
    from dotenv import load_dotenv
    load_dotenv(REPO / ".env")
except ImportError:
    pass
sys.path.insert(0, str(REPO / "src"))

import timesfm

RNG_SEED = 42
N = 100
K_FIRST = 11
rng = np.random.default_rng(RNG_SEED)

/Users/nikhileshbelulkar/Documents/timesfm_google/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Simulate Markov-switching AR(1)

Build **s[0], …** from the transition matrix, then update **y[t]** in a simple loop.

In [2]:
def simulate_markov_switching_ar1(
    n: int,
    rng: np.random.Generator,
) -> np.ndarray:
    # P[i,j] = P(s_t = j | s_{t-1} = i)
    P = np.array([[0.93, 0.07], [0.12, 0.88]])
    phi = np.array([0.48, -0.36])
    sigma = np.array([1.0, 1.15])

    s = np.zeros(n, dtype=np.int64)
    for t in range(1, n):
        s[t] = rng.choice(2, p=P[s[t - 1]])

    y = np.zeros(n, dtype=np.float64)
    y[0] = rng.normal(0.0, 1.0)
    for t in range(1, n):
        eta = rng.normal(0.0, 1.0)
        y[t] = phi[s[t]] * y[t - 1] + sigma[s[t]] * eta
    return y


y = simulate_markov_switching_ar1(N, rng)

## 3. One-step AR(1) forecast (OLS benchmark)

Regress **y[1:]** on **intercept** and **y[:-1]**; one-step prediction is **a + b · y_{t-1}** when the history ends at **t−1** (length **k** means we predict the next value after **y[k−1]**).

In [3]:
def forecast_ar1_ols(history: np.ndarray) -> float | None:
    try:
        if len(history) < 8:
            return None
        x = history[:-1]
        y_dep = history[1:]
        A = np.column_stack([np.ones(len(x)), x])
        beta, _, _, _ = np.linalg.lstsq(A, y_dep, rcond=None)
        a, b = float(beta[0]), float(beta[1])
        return a + b * float(history[-1])
    except Exception:
        raise Exception

## 4. Load TimesFM (once)

**horizon = 1** in the loop.

In [4]:
torch.set_float32_matmul_precision("high")
tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch",
    torch_compile=False,
)
tfm.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=256,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=False,
        fix_quantile_crossing=True,
    )
)

## 5. Expanding-window loop and MSE

For **k = 11, …, 99**: compare **OLS AR(1)** vs **TimesFM**.

In [5]:
rows: list[dict] = []
test_ks = list(range(K_FIRST, N))
se_ols: list[float] = []
se_tf: list[float] = []
n_fail = 0

for k in test_ks:
    actual = float(y[k])
    hist = y[:k].astype(np.float32)
    pred_tf = float(tfm.forecast(horizon=1, inputs=[hist])[0][0, 0])
    pred_ols = forecast_ar1_ols(y[:k])
    e_ols = actual - pred_ols
    e_tf = actual - pred_tf
    se_ols.append(e_ols**2)
    se_tf.append(e_tf**2)

rows.append(
    {
        "n": N,
        "k_first": K_FIRST,
        "n_test": len(test_ks),
        "ols_ar1_failures": n_fail,
        "mse_ols_ar1": float(np.mean(se_ols)) if se_ols else float("nan"),
        "mse_timesfm": float(np.mean(se_tf)) if se_tf else float("nan"),
    }
)

results = pd.DataFrame(rows)
results

,n,k_first,n_test,ols_ar1_failures,mse_ols_ar1,mse_timesfm
0,100,11,89,0,1.254519,1.30623


## 6. Save table

CSV under `output/`.

In [6]:
out_dir = HERE / "output"
out_dir.mkdir(parents=True, exist_ok=True)
path = out_dir / "hamilton_switching_vs_timesfm_results.csv"
results.to_csv(path, index=False)
print(path)

/Users/nikhileshbelulkar/Documents/timesfm_google/experiments_nikhilesh_experiments/output/hamilton_switching_vs_timesfm_results.csv
